# 01_extract_text_and_countries

In [1]:
import spacy
from spacy.ml import Doc
import pymupdf
import re
import csv
import os

In [2]:
DATA_FOLDER = "../data/"
PDF_FOLDER = f"{DATA_FOLDER}pdf_data/"
TXT_FOLDER = f"{DATA_FOLDER}txt_data/"
CSV_ENTITIES_FOLDER = f"{DATA_FOLDER}csv_entities/"
EMAIL_REGEX = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"

# Load English tokenizer, tagger, parser and NER
nlp_model = spacy.load("en_core_web_sm")

## PDF extraction
I am using PyMuPDF to extract text from PDF.

In [3]:
def extract_text_from_pdf(file_name: str):
    print(f"Extracting text from {PDF_FOLDER}{file_name}.pdf...")
    doc = pymupdf.open(f"{PDF_FOLDER}{file_name}.pdf") # open a document
    out = open(f"{TXT_FOLDER}{file_name}.txt", "wb") # create a text output
    for page in doc: # iterate the document pages
        extracted_text = page.get_text().encode("utf8") # get plain text (is in UTF-8)
        out.write(extracted_text) # write text of page
        out.write(bytes((12,))) # write page delimiter (form feed 0x0C)
    print(f"Text extracted and saved to {TXT_FOLDER}{file_name}.txt.")
    out.close()

## Text cleaning

In [4]:
def clear_text(file_name: str):
    file_path = f"{TXT_FOLDER}{file_name}.txt"
    print(f"Cleaning text in {file_path}.txt...")

    with open(file_path, "r") as file:
        text = file.read()

    cleaned_text = re.sub(EMAIL_REGEX, "", text)
    
    with open(file_path, "w") as file:
        file.write(cleaned_text)
    print(f"Text cleaned and saved to {file_path}.")

## Text processing

In [5]:
def process_doc(file_name: str) -> Doc:
    file_path = f"{TXT_FOLDER}{file_name}.txt"
    print(f"Processing document {file_path} with NLP model...")

    with open(file_path, "r") as file:
        text = file.read()

    doc = nlp_model(text)
    print("Document processed!")
    return doc

## Dataset building

- NORP: Nationalities, religious or political groups
- GPE: Geopolitical entities (countries, cities, states, etc.)

In [6]:
def build_actors_csv(doc: Doc, file_name: str):
    csv_file_path = f"{CSV_ENTITIES_FOLDER}{file_name}.csv"
    print(f"Building actors CSV file at {csv_file_path}...")
    actors = set()
    for ent in doc.ents:
        if ent.label_ in ["GPE", "NORP"]:
            actors.add((ent.text, ent.label_, "", ""))

    with open(csv_file_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Entity', 'Label', 'Validity', 'Mapped label'])
        writer.writerows(actors)
    print(f"Actors CSV file built at {csv_file_path}.")

In [7]:
for pdf in os.listdir(PDF_FOLDER):
    file_name = os.path.basename(pdf)
    print(f"Found file: {file_name}")
    if file_name == ".DS_Store":
        continue

    FILE_NAME = file_name.split(".")[0]
    print(f"Processing {FILE_NAME}...")
    
    extract_text_from_pdf(FILE_NAME)
    clear_text(FILE_NAME)
    processed_doc = process_doc(FILE_NAME)
    build_actors_csv(processed_doc, FILE_NAME)

    print(f"{FILE_NAME} processed!\n")

Found file: 1999.pdf
Processing 1999...
Extracting text from ../data/pdf_data/1999.pdf...
Text extracted and saved to ../data/txt_data/1999.txt.
Cleaning text in ../data/txt_data/1999.txt.txt...
Text cleaned and saved to ../data/txt_data/1999.txt.
Processing document ../data/txt_data/1999.txt with NLP model...
Document processed!
Building actors CSV file at ../data/csv_entities/1999.csv...
Actors CSV file built at ../data/csv_entities/1999.csv.
1999 processed!

Found file: 1998.pdf
Processing 1998...
Extracting text from ../data/pdf_data/1998.pdf...
Text extracted and saved to ../data/txt_data/1998.txt.
Cleaning text in ../data/txt_data/1998.txt.txt...
Text cleaned and saved to ../data/txt_data/1998.txt.
Processing document ../data/txt_data/1998.txt with NLP model...
Document processed!
Building actors CSV file at ../data/csv_entities/1998.csv...
Actors CSV file built at ../data/csv_entities/1998.csv.
1998 processed!

Found file: 2001.pdf
Processing 2001...
Extracting text from ../data